# RSQR repo import notebook

This is the minimal notebook you can upload to Colab and run to import the repo, then load the Qwen 1.5B benchmark code from the project itself.

No heavy benchmark cells, no long output, no extra fluff. Just the repo path fix and the import path you need.

In [ ]:
# Colab setup: install the minimum packages needed for the repo and model import path.
!python -V
!pip install -q --upgrade pip
!pip install -q transformers accelerate sentencepiece

import os
import sys
from pathlib import Path

repo_candidates = [
    Path('/content/kv-eviction'),
    Path('/workspace/kv-eviction'),
    Path.cwd() / 'kv-eviction',
    Path.cwd(),
]

repo_root = None
for candidate in repo_candidates:
    if candidate.exists() and (candidate / 'src').exists():
        repo_root = candidate
        break

if repo_root is None:
    possible = sorted(Path.cwd().glob('**/src'), key=lambda p: len(p.parts))
    if possible:
        repo_root = possible[0].parent

if repo_root is None:
    raise FileNotFoundError('Repo not found. Clone or mount the project to /content/kv-eviction or /workspace/kv-eviction.')

sys.path.insert(0, str(repo_root))
print('repo_root =', repo_root)

from src.eviction import EvictionManager, WindowState
from src.index_map import IndexMap
from src.rope import apply_rope, precompute_rope_freqs
from src.shadow_cache import ShadowCache

print('imports_ok = True')

In [ ]:
# Optional: quick smoke check if you want to confirm the repo package loads correctly.
import torch

freqs = precompute_rope_freqs(1024, 8, device=torch.device('cpu'))
raw_key = torch.randn(8, dtype=torch.float32)
shadow_cache = ShadowCache(survivor_every=8)
shadow_cache.add(token_id=7, raw_key=raw_key, rotated_key=raw_key.clone(), is_survivor=True)
index_map = IndexMap()
index_map.compact([7])
manager = EvictionManager(freqs)
boundary = manager.on_boundary(shadow_cache, index_map, WindowState(window_size=64, evict_n=8))
rotated = boundary['rotated'][0]['key']
expected = apply_rope(
    raw_key.unsqueeze(0).unsqueeze(0),
    torch.tensor([0.0], dtype=torch.float32),
    freqs,
).squeeze(0).squeeze(0)
max_abs_diff = (rotated - expected).abs().max().item()
print('smoke_max_abs_diff =', max_abs_diff)
assert max_abs_diff < 1e-5, max_abs_diff

In [ ]:
# This is the actual benchmark cell you will run in Colab with Qwen 1.5B.
# Keep it in a separate cell so you can use it only when you want the real test.
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'
print('loading model ...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print('model_loaded = True')

## Use this notebook as the upload-and-run bridge

- Mount or clone the repo in Colab.
- Run the first cell to fix the repo import path.
- Run the model-loading cell when you want to test the Qwen 1.5B setup.
- Do not run the heavy benchmark logic on your laptop; run the notebook on Colab.